# Data Cleaning - Olist E-Commerce

Notebook ini menangani masalah kualitas data yang ditemukan di tahap eksplorasi.

**Langkah cleaning:**
1. Filter order berdasarkan status sebelum drop NULL
2. Hapus baris dengan nilai kosong pada kolom kritis di products
3. Hapus produk dengan berat tidak valid (0 g)
4. Konversi tipe data (tanggal dan numerik)
5. Validasi hasil dan simpan data bersih

**Prasyarat:** Jalankan `01_load_to_sqlite.ipynb` terlebih dahulu.

## Persiapan

In [9]:
import os
import sqlite3

import pandas as pd

In [10]:
DB_PATH = "../data/database/olist.db"
CLEAN_DIR = "../data/clean/"

os.makedirs(CLEAN_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

## Muat Data Mentah

In [11]:
orders = pd.read_sql_query("SELECT * FROM orders", conn)
order_item = pd.read_sql_query("SELECT * FROM order_item", conn)
products = pd.read_sql_query("SELECT * FROM products", conn)
customers = pd.read_sql_query("SELECT * FROM customers", conn)

conn.close()

print(f"orders:     {orders.shape}")
print(f"order_item: {order_item.shape}")
print(f"products:   {products.shape}")
print(f"customers:  {customers.shape}")

orders:     (99441, 8)
order_item: (112650, 7)
products:   (32951, 9)
customers:  (99441, 5)


---
## 1. Cleaning: Tabel orders

Dari eksplorasi ditemukan:
- `order_approved_at`: 160 NULL
- `order_delivered_carrier_date`: 1.783 NULL
- `order_delivered_customer_date`: 2.965 NULL

NULL ini wajar untuk order yang belum selesai (canceled, shipped, processing, dll).
Strategi: filter hanya order dengan status `delivered`, lalu drop sisa NULL yang
seharusnya tidak ada.

In [12]:
# Distribusi status sebelum filter
print("Sebelum filter:")
print(orders["order_status"].value_counts())
print(f"\nTotal: {len(orders)}")

Sebelum filter:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Total: 99441


In [13]:
# Filter hanya order delivered
orders = orders[orders["order_status"] == "delivered"].copy()

print(f"Setelah filter delivered: {len(orders)}")

Setelah filter delivered: 96478


In [14]:
# Cek apakah masih ada NULL setelah filter
print("NULL setelah filter delivered:")
print(orders.isnull().sum())

NULL setelah filter delivered:
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [15]:
# Drop baris yang masih punya NULL (seharusnya tidak banyak)
sebelum = len(orders)
orders = orders.dropna().copy()
sesudah = len(orders)

print(f"Baris dihapus: {sebelum - sesudah}")
print(f"Sisa: {sesudah}")

Baris dihapus: 23
Sisa: 96455


In [16]:
# Konversi kolom tanggal ke datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

---
## 2. Cleaning: Tabel products

Dari eksplorasi ditemukan:
- 610 produk tanpa `product_category_name` (dan kolom terkait: name_length, desc_length, photos_qty)
- 2 produk tanpa `product_weight_g` dan dimensi
- 4 produk dengan berat = 0 g (tidak valid)

In [17]:
print("NULL sebelum cleaning:")
print(products.isnull().sum())
print(f"\nBerat = 0: {(products['product_weight_g'] == 0).sum()}")

NULL sebelum cleaning:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Berat = 0: 4


In [18]:
# Drop produk tanpa kategori dan tanpa dimensi/berat
sebelum = len(products)
products = products.dropna(
    subset=["product_category_name", "product_weight_g"]
).copy()
sesudah = len(products)

print(f"Baris dihapus (NULL): {sebelum - sesudah}")

Baris dihapus (NULL): 611


In [19]:
# Hapus produk dengan berat 0 g
sebelum = len(products)
products = products[products["product_weight_g"] > 0].copy()
sesudah = len(products)

print(f"Baris dihapus (berat 0): {sebelum - sesudah}")
print(f"Sisa: {sesudah}")

Baris dihapus (berat 0): 4
Sisa: 32336


In [20]:
# Konversi kolom numerik yang seharusnya integer
int_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
]

for col in int_cols:
    products[col] = products[col].astype(int)

products.dtypes

product_id                        str
product_category_name             str
product_name_lenght             int64
product_description_lenght      int64
product_photos_qty              int64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

---
## 3. Cleaning: Tabel order_item

Dari eksplorasi tidak ditemukan masalah (tidak ada NULL, tidak ada harga negatif).
Yang perlu dilakukan: konversi tipe data tanggal.

In [21]:
order_item["shipping_limit_date"] = pd.to_datetime(order_item["shipping_limit_date"])

order_item.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

---
## 4. Cleaning: Tabel customers

Dari eksplorasi tidak ditemukan masalah (tidak ada NULL, tidak ada duplikat).
Tidak perlu tindakan.

In [22]:
print(f"customers: {customers.shape}")
print(f"NULL: {customers.isnull().sum().sum()}")

customers: (99441, 5)
NULL: 0


---
## 5. Sinkronisasi Antar Tabel

Setelah filter orders ke status `delivered`, baris di order_item dan customers
yang tidak lagi punya pasangan di orders perlu dihapus agar data tetap konsisten.

In [23]:
# Sinkronisasi order_item dengan orders
sebelum = len(order_item)
order_item = order_item[order_item["order_id"].isin(orders["order_id"])].copy()
print(f"order_item: {sebelum} -> {len(order_item)}")

# Sinkronisasi order_item dengan products
sebelum = len(order_item)
order_item = order_item[order_item["product_id"].isin(products["product_id"])].copy()
print(f"order_item (setelah sync products): {sebelum} -> {len(order_item)}")

# Sinkronisasi customers dengan orders
sebelum = len(customers)
customers = customers[customers["customer_id"].isin(orders["customer_id"])].copy()
print(f"customers: {sebelum} -> {len(customers)}")

order_item: 112650 -> 110173
order_item (setelah sync products): 110173 -> 108628
customers: 99441 -> 96455


---
## 6. Validasi Hasil Cleaning

In [24]:
print("Dimensi tabel setelah cleaning:")
print(f"  orders:     {orders.shape}")
print(f"  order_item: {order_item.shape}")
print(f"  products:   {products.shape}")
print(f"  customers:  {customers.shape}")

print("\nTotal NULL per tabel:")
for name, df in [("orders", orders), ("order_item", order_item), ("products", products), ("customers", customers)]:
    null_count = df.isnull().sum().sum()
    print(f"  {name}: {null_count}")

Dimensi tabel setelah cleaning:
  orders:     (96455, 8)
  order_item: (108628, 7)
  products:   (32336, 9)
  customers:  (96455, 5)

Total NULL per tabel:
  orders: 0
  order_item: 0
  products: 0
  customers: 0


In [25]:
# Cek integritas relasi setelah cleaning
oi_tanpa_order = order_item[~order_item["order_id"].isin(orders["order_id"])]
oi_tanpa_produk = order_item[~order_item["product_id"].isin(products["product_id"])]
o_tanpa_customer = orders[~orders["customer_id"].isin(customers["customer_id"])]

print("Integritas relasi:")
print(f"  order_item tanpa order:   {len(oi_tanpa_order)}")
print(f"  order_item tanpa produk:  {len(oi_tanpa_produk)}")
print(f"  orders tanpa customer:    {len(o_tanpa_customer)}")

Integritas relasi:
  order_item tanpa order:   0
  order_item tanpa produk:  0
  orders tanpa customer:    0


---
## 7. Simpan Data Bersih

In [26]:
conn = sqlite3.connect(DB_PATH)

orders.to_sql("orders", conn, if_exists="replace", index=False)
order_item.to_sql("order_item", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

conn.close()

print("Data bersih disimpan ke SQLite.")

Data bersih disimpan ke SQLite.


In [27]:
# Simpan juga ke CSV untuk referensi
orders.to_csv(os.path.join(CLEAN_DIR, "orders.csv"), index=False)
order_item.to_csv(os.path.join(CLEAN_DIR, "order_item.csv"), index=False)
products.to_csv(os.path.join(CLEAN_DIR, "products.csv"), index=False)
customers.to_csv(os.path.join(CLEAN_DIR, "customers.csv"), index=False)

print(f"Data bersih disimpan ke {CLEAN_DIR}")

Data bersih disimpan ke ../data/clean/


---
## Ringkasan

| Tabel | Sebelum | Sesudah | Dihapus | Tindakan |
|-------|---------|---------|---------|----------|
| orders | 99.441 | 96.455 | 2.986 | Filter status delivered, drop 23 sisa NULL, konversi datetime |
| order_item | 112.650 | 108.628 | 4.022 | Sinkronisasi dengan orders (-2.477) dan products (-1.545), konversi datetime |
| products | 32.951 | 32.336 | 615 | Drop 611 NULL (kategori/berat), hapus 4 berat 0 g, konversi float ke int |
| customers | 99.441 | 96.455 | 2.986 | Sinkronisasi dengan orders |

Data bersih tersimpan di:
- SQLite: `data/database/olist.db`
- CSV: `data/clean/*.csv`